## Setup
Imports, particle masses and helper functions used throughout the notebook.

In [ ]:
import pickle
import numpy as np
import pandas
from matplotlib import pyplot as plt

#Particle masses in MeV/c^2 (PDG values)
M_KAON  = 493.7
M_MUON  = 105.7
M_PION  = 139.6
M_JPSI  = 3097
M_PSI2S = 3686

In [ ]:
def track_energy(data, track, mass):
    #Energy of a track given the mass we choose. mass=None uses the energy stored in the data.
    if mass is None:
        return data[f'{track} 4-momentum energy component']
    p = [data[f'{track} 4-momentum {c} component'] for c in 'xyz']
    return np.sqrt(p[0]**2 + p[1]**2 + p[2]**2 + mass**2)


def two_body_mass(data, track1, track2, mass1=None, mass2=None):
    #Invariant mass of two tracks, where we choose which mass each track is given.
    #track names are 'Kaon', 'Opposite-sign muon' or 'Same-sign muon'.
    #Changing mass1/mass2 lets us test what happens if a particle was misidentified.
    p1 = [data[f'{track1} 4-momentum {c} component'] for c in 'xyz']
    p2 = [data[f'{track2} 4-momentum {c} component'] for c in 'xyz']

    energy1 = track_energy(data, track1, mass1)
    energy2 = track_energy(data, track2, mass2)

    combined_energy = energy1 + energy2
    combined_momentum_sq = (p1[0] + p2[0])**2 + (p1[1] + p2[1])**2 + (p1[2] + p2[2])**2
    return np.sqrt(combined_energy**2 - combined_momentum_sq)


def plot_mass(values, xlabel, title=None, show_peak=False, bins=100, **hist_kwargs):
    #Histogram of a mass, with the y-axis label showing the real bin width.
    #show_peak=True marks the centre of the tallest bin.
    counts, edges, _ = plt.hist(values, bins=bins, **hist_kwargs)
    bin_width = edges[1] - edges[0]

    if show_peak:
        i = np.argmax(counts)
        peak = 0.5 * (edges[i] + edges[i + 1])
        plt.axvline(peak, linestyle='--', color='red', label=f'Peak: {peak:.0f} MeV')
        plt.legend()

    if title:
        plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(rf'Candidates / ({bin_width:.0f} MeV/$c^2$)')
    plt.show()

In [ ]:
#Open the data file
with open('../Data/dataset_2011.pkl', 'rb') as infile:
  data_2011 = pickle.load(infile)

#Shortcuts for the columns we use the most
b_mass      = data_2011['B invariant mass']
dimuon_mass = data_2011['dimuon-system invariant mass']

In [ ]:
#We can look at all the types of information stored in this data
list(data_2011)

## Task 1: B mass distribution and fit

In [ ]:
import zfit

lower, upper = 5000, 6000
b_mass_space = zfit.Space('B invariant mass', (lower, upper))

mean  = zfit.Parameter('mean', 5280, 5000, 6000)
sigma = zfit.Parameter('sigma', 20,    5,  100)
signalYield = zfit.Parameter('signalYield', 1e5, 0, 1e6)

signalPDF = zfit.pdf.Gauss(obs=b_mass_space, mu=mean, sigma=sigma, extended=signalYield)

exponent  = zfit.Parameter('exponent', -0.01, -1e-1, -1e-6)
backgroundYield = zfit.Parameter('backgroundYield', 4e5, 0, 1e6)

backgroundPDF = zfit.pdf.Exponential(obs=b_mass_space, lam=exponent, extended=backgroundYield)

totalPDF = zfit.pdf.SumPDF(pdfs=[signalPDF, backgroundPDF])

zdata = zfit.Data.from_pandas(data_2011, b_mass_space)

nll = zfit.loss.ExtendedUnbinnedNLL(model=totalPDF, data=zdata)

minimizer = zfit.minimize.Minuit()
result = minimizer.minimize(nll)

#Uncertainties from Hesse (the curvature of the likelihood at the minimum)
param_hesse = result.hesse()

mu_fit         = result.params[mean]['value']
sigma_fit      = result.params[sigma]['value']
sig_yield_fit  = result.params[signalYield]['value']
back_yield_fit = result.params[backgroundYield]['value']
exponent_fit   = result.params[exponent]['value']


print(result)
print(f"Function minimum: {result.fmin}")
print(f"Converged: {result.converged}")
print(f"Valid: {result.valid}")


n_bins = 100
bin_width = (upper - lower) / n_bins

plt.hist(b_mass, bins=n_bins, range=(lower, upper),
          color='blue', label='Data')

x = np.linspace(lower, upper, 1000)
plt.plot(x, np.asarray(totalPDF.ext_pdf(x))      * bin_width, 'r-',  label='Total fit')
plt.plot(x, np.asarray(signalPDF.ext_pdf(x))     * bin_width, 'y--', label='Signal (Gaussian)')
plt.plot(x, np.asarray(backgroundPDF.ext_pdf(x)) * bin_width, 'g:',  label='Background (exponential)')

plt.xlabel(r'B candidate mass / MeV/$c^2$')
plt.ylabel(rf'Candidates / ({bin_width:.0f} MeV/$c^2$)')
plt.legend()
plt.show()

In [ ]:
#Most of the real B mesons are actually B -> J/psi K decays. The J/psi is a charm-anticharm meson which decays to our two muons, and
#so, the dimuon system will have a mass consistent with the J/psi meson, 3097 MeV. If we remove these we get

not_jpsi = abs(dimuon_mass - M_JPSI) > 100

plot_mass(b_mass[not_jpsi], r'B candidate mass / MeV/$c^2$')

In [ ]:
#Now most of what's remaining are actually B -> psi(2S) K decays. The psi(2S) is an excited state of the J/psi.
#It can also decay to our two muons, but now the dimuon system will have a mass consistent with the psi(2S) meson, 3686 MeV.
#If we also remove these, we'll see how much work we've got to do

not_psi2s = abs(dimuon_mass - M_PSI2S) > 100

plot_mass(b_mass[not_jpsi & not_psi2s], r'B candidate mass / MeV/$c^2$')

## Task 2: Dimuon mass spectrum

In [ ]:
#J/psi and psi(2S) regions of the dimuon spectrum
jpsi_region  = data_2011[abs(dimuon_mass - M_JPSI)  < 50]
psi2s_region = data_2011[abs(dimuon_mass - M_PSI2S) < 50]

plot_mass(jpsi_region['dimuon-system invariant mass'],  r'Dimuon mass / MeV/$c^2$', title=r'J/$\psi$ peak')
plot_mass(psi2s_region['dimuon-system invariant mass'], r'Dimuon mass / MeV/$c^2$', title=r'$\psi(2S)$ peak')
plot_mass(dimuon_mass, r'Dimuon mass / MeV/$c^2$', title='Full dimuon spectrum')

## Task 3: Signal-like and background-like regions
### Signal-like data

In [ ]:
#The J/psi region is almost all real B -> J/psi K decays
signal_like = jpsi_region

### Background-like data

In [ ]:
#Nothing real can have a mass this far above the B mass, so this is all random combinations
noise_like = data_2011[b_mass > 5700]

plot_mass(noise_like['B invariant mass'], r'B candidate mass / MeV/$c^2$', title='Background-like data: B mass > 5700 MeV')

### Identify how features vary between datasets

In [ ]:
#(variable, plot range)
compare_variables = [
    ('B flight distance chi2 wrt primary vertex',  (0, 5000)),
    ('B impact parameter chi2 wrt primary vertex', (0, 10)),
]

for variable, plot_range in compare_variables:
    plt.hist(signal_like[variable], bins=100, range=plot_range, label='Signal-like', density=True)
    plt.hist(noise_like[variable],  bins=100, range=plot_range, label='Noise-like',  density=True)
    plt.xlabel(variable)
    plt.ylabel('Candidates (normalised)')
    plt.legend()
    plt.show()

In [ ]:
#Correlation of every variable with the B mass. Cutting on a variable that is strongly
#correlated with the B mass would change the shape of the mass distribution.

import scipy.stats as stats
from tabulate import tabulate

r_list = []
for var in data_2011:
    r = np.abs(stats.pearsonr(data_2011[var], b_mass).statistic)
    r_list.append((var, r))

r_list = sorted(r_list, key=lambda x: -np.inf if np.isnan(x[1]) else x[1], reverse=True)
print(tabulate(r_list, headers=['Variable', 'PMCC']))

## Task 4: Selection with XGBoost
### Training

In [ ]:
import xgboost

model = xgboost.XGBClassifier(eval_metric='auc', early_stopping_rounds=50, n_estimators=1000, learning_rate=0.3, max_depth=6)

nSignalTotal = len(signal_like)
nBackgroundTotal = len(noise_like)

#vary these
chosen_variables = [
'B decay vertex x position',
'B decay vertex y position',
'B decay vertex z position',
'B decay vertex fit chi2',
'B impact parameter wrt primary vertex',
'B impact parameter chi2 wrt primary vertex',
'B flight distance wrt primary vertex',
'B flight distance chi2 wrt primary vertex',
'B cos(angle) between line-of-flight and momentum',
'B magnitude of momentum'
]

#train/test split
nSignalTrain = int(0.9*nSignalTotal)
nSignalTest  = nSignalTotal - nSignalTrain
nBackgroundTrain = int(0.9*nBackgroundTotal)
nBackgroundTest  = nBackgroundTotal - nBackgroundTrain

#train
x_train = pandas.concat([signal_like[chosen_variables][:nSignalTrain], noise_like[chosen_variables][:nBackgroundTrain]], axis=0, ignore_index=True)
y_train = [1]*nSignalTrain + [0]*nBackgroundTrain

#test
x_test = pandas.concat([signal_like[chosen_variables][nSignalTrain:], noise_like[chosen_variables][nBackgroundTrain:]], axis=0, ignore_index=True)
y_test  = [1]*nSignalTest + [0]*nBackgroundTest

model.fit(x_train, y_train, eval_set=[(x_test,y_test)])

### Results on the test sample

In [ ]:
from sklearn.metrics import roc_curve

test_prediction = model.predict_proba(x_test)

plt.hist([p[1] for p,cls in zip(test_prediction, y_test) if cls == 1], bins=100, label='Signal')
plt.hist([p[1] for p,cls in zip(test_prediction, y_test) if cls == 0], bins=100, label='Background')
plt.xlabel(r'Signal Probability')
plt.ylabel(r'Candidates')
plt.legend()
plt.show()

background_accepted, signal_accepted, probabilities_tested = roc_curve(y_test, test_prediction[:,1])
plt.plot(background_accepted, signal_accepted, label='ROC curve')
plt.plot([0,1], [0,1], color='red', linestyle='--', label = 'Random')
plt.xlabel(r'Background Contamination')
plt.ylabel(r'Signal Efficiency')
plt.legend()
plt.show()

### Apply the selection to the full dataset

In [ ]:
#Signal probability for every candidate in the dataset
signal_probability = model.predict_proba(data_2011[chosen_variables])[:, 1]

#threshold set at 0.99
passes_bdt = signal_probability > 0.99

## Task 5: K+ mu- two body system

In [ ]:
#Using the energies stored in the data (the detector's own mass assignments)
kmu_mass = two_body_mass(data_2011, 'Kaon', 'Opposite-sign muon')

plot_mass(kmu_mass, r'Combined mass of $K^+$ and $\mu^-$ / MeV/$c^2$', show_peak=True)

### Masses correct
Kaon given the kaon mass, muon given the muon mass.

In [ ]:
kmu_mass_correct = two_body_mass(data_2011, 'Kaon', 'Opposite-sign muon', M_KAON, M_MUON)

plot_mass(kmu_mass_correct, r'Combined mass of $K^+$ and $\mu^-$ / MeV/$c^2$', show_peak=True)

## Kaon mistaken for a muon (so muon has kaon mass)

In [ ]:
kmu_mass_as_kk = two_body_mass(data_2011, 'Kaon', 'Opposite-sign muon', M_KAON, M_KAON)

plot_mass(kmu_mass_as_kk, r'Combined mass, both tracks given the kaon mass / MeV/$c^2$', show_peak=True)

## Muon mistaken for a kaon (so kaon has muon mass)

In [ ]:
kmu_mass_as_mumu = two_body_mass(data_2011, 'Kaon', 'Opposite-sign muon', M_MUON, M_MUON)

#before the XGBoost selection
plot_mass(kmu_mass_as_mumu, r'Combined mass, both tracks given the muon mass / MeV/$c^2$', show_peak=True)

#after the XGBoost selection
plot_mass(kmu_mass_as_mumu[passes_bdt], r'Combined mass, both tracks given the muon mass / MeV/$c^2$', show_peak=True)

## Kill the peak

after the XGBoost algorithm is used to cut out the background noise the mode is exactly 3097. In general, the peak being close to 3000 MeV/c^2 seems to be a coincidence (from the research that I've done. So a peak within that region or so is normal.). However, the peak at 3097 specifically is not normal. 

We previously cleaned the J/psi decays. However, that assumes muons are identified correctly. If one of the muons registered as a kaon in the detector then the 3097 J/psi decay would register as a 3097 kaon + muon combination. This is the peak we observe here. 

As a temporary solution (to be refined later) we will simply cut out 3097+-50 from this kaon+muon combination. While not perfect, it gets rid off a lot of bad data while not sacrificing too much. A more refined solution involving fits or maybe ML should be implemented at some point

In [ ]:
#Veto candidates where the kaon + muon look like a J/psi (the kaon was really a muon)
jpsi_swap_veto = abs(kmu_mass_as_mumu - M_JPSI) > 50

plot_mass(kmu_mass_as_mumu[passes_bdt & jpsi_swap_veto], r'Combined mass, both tracks given the muon mass / MeV/$c^2$')

## Task 6: Finalising the B count

In [ ]:
#All cuts for the rare B -> K mu mu decay together
keep = (not_jpsi            #dimuon not a J/psi
    & not_psi2s             #dimuon not a psi(2S)
    & passes_bdt            #XGBoost selection
    & jpsi_swap_veto        #kaon + muon not a J/psi
)

plot_mass(b_mass[keep], r'B candidate mass / MeV/$c^2$')